In [ ]:
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# KG structure
kg_triples = pd.read_parquet("kg_triples_ids.parquet")
entity2id = pickle.load(open("entity2id.pkl", "rb"))
relation2id = pickle.load(open("relation2id.pkl", "rb"))

# Precomputed embeddings
title_embeddings = np.load("title_embeddings.npy")
image_embeddings = np.load("image_embeddings.npy")
image_item_ids = np.load("image_item_ids.npy", allow_pickle=True)

num_entities = len(entity2id)
num_relations = len(relation2id)

print(f"Entities: {num_entities}")
print(f"Relations: {num_relations}")
print(f"Triples: {len(kg_triples)}")
print(f"Title embeddings: {title_embeddings.shape}")
print(f"Image embeddings: {image_embeddings.shape}")

# Verify triples have confidence
assert "confidence" in kg_triples.columns, "Missing confidence column!"
print(f"Confidence range: [{kg_triples['confidence'].min()}, {kg_triples['confidence'].max()}]")

In [ ]:
import gzip
import json
import pandas as pd

def stream_json_gz(path, domain_name, max_items=None):
    data = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_items and i >= max_items:
                break
            record = json.loads(line)
            record["domain"] = domain_name
            data.append(record)
    return pd.DataFrame(data)

electronics = stream_json_gz(
    "/DATA/shourya_2211mc14/Sougata2/mmprodnet/datasets/meta_Electronics.jsonl.gz",
    domain_name="electronics",
    max_items=200000
)
clothing = stream_json_gz(
    "/DATA/shourya_2211mc14/Sougata2/mmprodnet/datasets/meta_Clothing_Shoes_and_Jewelry.jsonl.gz",
    domain_name="clothing",
    max_items=200000
)
home = stream_json_gz(
    "/DATA/shourya_2211mc14/Sougata2/mmprodnet/datasets/meta_Home_and_Kitchen.jsonl.gz",
    domain_name="home",
    max_items=200000
)

df = pd.concat([electronics, clothing, home], ignore_index=True)

# MUST apply same standardization as original notebook
df = df.rename(columns={"parent_asin": "item_id"})

columns_to_keep = [
    "item_id", "title", "categories", "details", "store",
    "main_category", "average_rating", "rating_number",
    "features", "description", "images", "domain"
]
df = df[columns_to_keep]
df = df.dropna(subset=["item_id", "title"])
df = df.reset_index(drop=True)

# Verify alignment with saved embeddings
title_embeddings = np.load("title_embeddings.npy")
assert len(df) == len(title_embeddings), \
    f"Mismatch: df has {len(df)} rows, embeddings have {len(title_embeddings)}"

print(f"Reconstructed df: {len(df)} rows")
print("Alignment with embeddings confirmed")

In [ ]:
# Map each entity ID to its text and image embedding
# df must still be in memory from the KG construction notebook
# If not, reload: df = pd.read_parquet("your_dataframe.parquet") or reconstruct

item_id_to_row = {item_id: idx for idx, item_id in enumerate(df["item_id"])}
image_id_to_pos = {str(item_id): pos for pos, item_id in enumerate(image_item_ids)}

TEXT_DIM = title_embeddings.shape[1]   # 768
IMAGE_DIM = image_embeddings.shape[1]  # 512

# Pre-allocate aligned matrices
text_emb_matrix = np.zeros((num_entities, TEXT_DIM), dtype=np.float32)
image_emb_matrix = np.zeros((num_entities, IMAGE_DIM), dtype=np.float32)
has_text = np.zeros(num_entities, dtype=np.bool_)
has_image = np.zeros(num_entities, dtype=np.bool_)

id2entity = {v: k for k, v in entity2id.items()}

for eid in range(num_entities):
    entity_name = id2entity[eid]

    if not entity_name.startswith("item_"):
        continue

    asin = entity_name[5:]  # strip "item_" prefix

    # Text embedding
    if asin in item_id_to_row:
        row_idx = item_id_to_row[asin]
        text_emb_matrix[eid] = title_embeddings[row_idx]
        has_text[eid] = True

    # Image embedding
    if asin in image_id_to_pos:
        img_pos = image_id_to_pos[asin]
        image_emb_matrix[eid] = image_embeddings[img_pos]
        has_image[eid] = True

print(f"Entities with text: {has_text.sum()}")
print(f"Entities with image: {has_image.sum()}")
print(f"Entities with both: {(has_text & has_image).sum()}")
print(f"Non-item entities (no modality): {(~has_text).sum()}")

In [ ]:
# Convert to torch tensors
text_emb_tensor = torch.tensor(text_emb_matrix, dtype=torch.float32)
image_emb_tensor = torch.tensor(image_emb_matrix, dtype=torch.float32)
has_text_tensor = torch.tensor(has_text, dtype=torch.bool)
has_image_tensor = torch.tensor(has_image, dtype=torch.bool)

# Free numpy arrays
del text_emb_matrix, image_emb_matrix, has_text, has_image
del title_embeddings, image_embeddings, image_item_ids
import gc
gc.collect()

print(f"Text tensor: {text_emb_tensor.shape}")
print(f"Image tensor: {image_emb_tensor.shape}")
print(f"GPU memory before model: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Convert triples to numpy arrays
triples = kg_triples[["head", "relation", "tail"]].values.astype(np.int64)
confidence = kg_triples["confidence"].values.astype(np.float32)

# Train/Validation/Test split (90/5/5)
np.random.seed(42)
num_triples = len(triples)
indices = np.random.permutation(num_triples)

test_size = int(0.05 * num_triples)
val_size = int(0.05 * num_triples)

test_idx = indices[:test_size]
val_idx = indices[test_size:test_size + val_size]
train_idx = indices[test_size + val_size:]

train_triples = triples[train_idx]
train_confidence = confidence[train_idx]
val_triples = triples[val_idx]
val_confidence = confidence[val_idx]
test_triples = triples[test_idx]
test_confidence = confidence[test_idx]

print(f"Train triples: {len(train_triples)}")
print(f"Val triples:   {len(val_triples)}")
print(f"Test triples:  {len(test_triples)}")

# Verify no leakage: check relation distribution is similar across splits
from collections import Counter
id2relation = {v: k for k, v in relation2id.items()}

print("\nRelation distribution across splits:")
for name, split in [("Train", train_triples), ("Val", val_triples), ("Test", test_triples)]:
    counts = Counter(split[:, 1])
    total = len(split)
    print(f"\n  {name}:")
    for rid in sorted(counts.keys()):
        print(f"    {id2relation[rid]:25s} {counts[rid]:>8d} ({counts[rid]/total*100:.1f}%)")

In [ ]:
class KGDataset(Dataset):
    def __init__(self, triples, confidence, num_entities, num_neg=128):
        self.triples = triples
        self.confidence = confidence
        self.num_entities = num_entities
        self.num_neg = num_neg

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        h, r, t = self.triples[idx]
        conf = self.confidence[idx]

        # Negative sampling: corrupt head or tail
        neg = np.random.randint(0, self.num_entities, size=self.num_neg)

        return (
            np.int64(h), np.int64(r), np.int64(t),
            np.float32(conf), neg.astype(np.int64)
        )


train_dataset = KGDataset(train_triples, train_confidence, num_entities, num_neg=64)
val_dataset = KGDataset(val_triples, val_confidence, num_entities, num_neg=64)
test_dataset = KGDataset(test_triples, test_confidence, num_entities, num_neg=64)

train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=4096, shuffle=False,
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False,
                         num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")



In [ ]:
class ModGatedRotatE(nn.Module):
    def __init__(self, num_entities, num_relations, dim,
                 text_dim, image_dim, text_emb, image_emb,
                 has_text, has_image, gamma=12.0, dropout=0.2):
        super().__init__()
        self.dim = dim
        self.gamma = nn.Parameter(torch.tensor(float(gamma)), requires_grad=False)
        self.epsilon = 2.0
        self.dropout = nn.Dropout(dropout)

        # Frozen modality embeddings
        self.register_buffer("text_emb", text_emb)
        self.register_buffer("image_emb", image_emb)
        self.register_buffer("has_text", has_text)
        self.register_buffer("has_image", has_image)

        # Trainable projectors
        self.text_proj = nn.Linear(text_dim, dim, bias=False)
        self.image_proj = nn.Linear(image_dim, dim, bias=False)

        # Learnable base entity embeddings
        self.entity_emb = nn.Embedding(num_entities, dim)

        # Relation embeddings: phase angles for RotatE
        self.relation_emb = nn.Embedding(num_relations, dim // 2)

        # Modality gate: one scalar per relation
        self.gate_weights = nn.Embedding(num_relations, 1)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.uniform_(self.relation_emb.weight, -np.pi, np.pi)
        nn.init.zeros_(self.gate_weights.weight)
        nn.init.xavier_uniform_(self.text_proj.weight)
        nn.init.xavier_uniform_(self.image_proj.weight)

    def get_entity_embedding(self, entity_ids, relation_ids):
        base = self.entity_emb(entity_ids)
        base = self.dropout(base)  # dropout on base embedding

        gate = torch.sigmoid(self.gate_weights(relation_ids))

        text_proj = self.text_proj(self.text_emb[entity_ids])
        image_proj = self.image_proj(self.image_emb[entity_ids])

        ht = self.has_text[entity_ids].unsqueeze(-1).float()
        hi = self.has_image[entity_ids].unsqueeze(-1).float()

        text_proj = text_proj * ht
        image_proj = image_proj * hi

        has_both = (ht * hi)
        text_only = ht * (1 - hi)

        modality = (
            has_both * (gate * text_proj + (1 - gate) * image_proj)
            + text_only * text_proj
        )

        modality = self.dropout(modality)  # dropout on modality

        return base + modality

    def entity_regularization(self):
        """L3 regularization on entity embeddings (from RotatE paper)."""
        return (self.entity_emb.weight.norm(p=3, dim=-1) ** 3).mean()

    def rotateE_score(self, head_emb, rel_emb, tail_emb):
        h_re, h_im = torch.chunk(head_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(tail_emb, 2, dim=-1)

        r_re = torch.cos(rel_emb)
        r_im = torch.sin(rel_emb)

        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re

        score = (hr_re - t_re).abs() + (hr_im - t_im).abs()

        return self.gamma - score.sum(dim=-1)

    def forward(self, head_ids, relation_ids, tail_ids):
        h = self.get_entity_embedding(head_ids, relation_ids)
        t = self.get_entity_embedding(tail_ids, relation_ids)
        r = self.relation_emb(relation_ids)

        return self.rotateE_score(h, r, t)

    def get_gates(self):
        with torch.no_grad():
            gates = torch.sigmoid(self.gate_weights.weight).squeeze().cpu().numpy()
        return gates

In [ ]:
EMBEDDING_DIM = 256
GAMMA = 12.0
DROPOUT = 0.2

model = ModGatedRotatE(
    num_entities=num_entities,
    num_relations=num_relations,
    dim=EMBEDDING_DIM,
    text_dim=TEXT_DIM,
    image_dim=IMAGE_DIM,
    text_emb=text_emb_tensor,
    image_emb=image_emb_tensor,
    has_text=has_text_tensor,
    has_image=has_image_tensor,
    gamma=GAMMA,
    dropout=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

In [ ]:
def self_adversarial_loss(model, head, relation, tail, confidence, neg_entities,
                          adversarial_temperature=1.0, reg_weight=0.001):
    batch_size = head.shape[0]
    num_neg = neg_entities.shape[1]

    # Positive scores
    pos_score = model(head, relation, tail)

    # Negative scores: corrupt tail
    neg_tail = neg_entities[:, :num_neg // 2]
    head_exp = head.unsqueeze(1).expand(-1, num_neg // 2)
    rel_exp = relation.unsqueeze(1).expand(-1, num_neg // 2)

    neg_tail_score = model(
        head_exp.reshape(-1),
        rel_exp.reshape(-1),
        neg_tail.reshape(-1)
    ).reshape(batch_size, num_neg // 2)

    # Negative scores: corrupt head
    neg_head = neg_entities[:, num_neg // 2:]
    tail_exp = tail.unsqueeze(1).expand(-1, num_neg // 2)

    neg_head_score = model(
        neg_head.reshape(-1),
        rel_exp.reshape(-1),
        tail_exp.reshape(-1)
    ).reshape(batch_size, num_neg // 2)

    neg_score = torch.cat([neg_tail_score, neg_head_score], dim=1)

    with torch.no_grad():
        neg_weights = F.softmax(neg_score * adversarial_temperature, dim=1)

    pos_loss = -F.logsigmoid(pos_score)
    neg_loss = -(neg_weights * F.logsigmoid(-neg_score)).sum(dim=1)

    # Confidence-weighted loss + entity regularization
    main_loss = (confidence * (pos_loss + neg_loss)).mean()
    reg_loss = model.entity_regularization()

    loss = main_loss + reg_weight * reg_loss

    return loss, pos_score.mean().item(), neg_score.mean().item()

In [ ]:
def train_epoch(model, loader, optimizer, adversarial_temp=1.0):
    model.train()
    total_loss = 0
    total_pos = 0
    total_neg = 0
    num_batches = 0

    for head, relation, tail, conf, neg in loader:
        head = head.to(device)
        relation = relation.to(device)
        tail = tail.to(device)
        conf = conf.to(device)
        neg = neg.to(device)

        optimizer.zero_grad()

        loss, pos_score, neg_score = self_adversarial_loss(
            model, head, relation, tail, conf, neg,
            adversarial_temperature=adversarial_temp,
            reg_weight=0.001
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_pos += pos_score
        total_neg += neg_score
        num_batches += 1

    return (
        total_loss / num_batches,
        total_pos / num_batches,
        total_neg / num_batches
    )


@torch.no_grad()
def validate(model, loader, adversarial_temp=1.0):
    model.eval()
    total_loss = 0
    total_pos = 0
    total_neg = 0
    num_batches = 0

    for head, relation, tail, conf, neg in loader:
        head = head.to(device)
        relation = relation.to(device)
        tail = tail.to(device)
        conf = conf.to(device)
        neg = neg.to(device)

        loss, pos_score, neg_score = self_adversarial_loss(
            model, head, relation, tail, conf, neg,
            adversarial_temperature=adversarial_temp,
            reg_weight=0.0  # no regularization during validation
        )

        total_loss += loss.item()
        total_pos += pos_score
        total_neg += neg_score
        num_batches += 1

    return (
        total_loss / num_batches,
        total_pos / num_batches,
        total_neg / num_batches
    )

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # use the free V100S

In [ ]:
NUM_EPOCHS = 150
LR = 3e-4           # lower LR
ADV_TEMP = 0.5       # lower temperature = softer negative weighting
PATIENCE = 20
REG_WEIGHT = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=7, factor=0.5
)

history = {
    "train_loss": [], "val_loss": [],
    "train_pos": [], "train_neg": [],
    "val_pos": [], "val_neg": [],
    "gates": []
}

best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    train_loss, train_pos, train_neg = train_epoch(
        model, train_loader, optimizer, ADV_TEMP
    )
    val_loss, val_pos, val_neg = validate(model, val_loader, ADV_TEMP)

    old_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]["lr"]
    if new_lr != old_lr:
        print(f"  LR reduced: {old_lr:.6f} → {new_lr:.6f}")

    elapsed = time.time() - start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_pos"].append(train_pos)
    history["train_neg"].append(train_neg)
    history["val_pos"].append(val_pos)
    history["val_neg"].append(val_neg)

    gates = model.get_gates()
    history["gates"].append(gates.tolist())

    if epoch % 5 == 0 or epoch == 1:
        gate_str = ", ".join([f"{id2relation[i]}={gates[i]:.3f}" for i in range(num_relations)])
        print(f"Epoch {epoch}/{NUM_EPOCHS} ({elapsed:.1f}s)")
        print(f"  Train loss: {train_loss:.4f} | pos: {train_pos:.3f} | neg: {train_neg:.3f}")
        print(f"  Val loss:   {val_loss:.4f} | pos: {val_pos:.3f} | neg: {val_neg:.3f}")
        print(f"  Gates: {gate_str}")
        print()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "modgated_rotate_best.pt")
        print(f"  → Saved best model (val_loss={val_loss:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            print(f"Best val_loss: {best_val_loss:.4f}")
            break

print(f"\nTraining complete. Ran {epoch} epochs. Best val_loss: {best_val_loss:.4f}")

In [ ]:
# ============================================================
# Training curve dynamics
# ============================================================
import matplotlib.pyplot as plt
import numpy as np, json

# Robust: use in-memory history, else reload the saved one
try:
    history
except NameError:
    history = json.load(open("rotateE_training_history.json"))

ep = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(2, 2, figsize=(14, 9))

# (1) Loss
ax[0,0].plot(ep, history["train_loss"], label="train")
ax[0,0].plot(ep, history["val_loss"],   label="val")
ax[0,0].set_title("Loss"); ax[0,0].set_xlabel("epoch")
ax[0,0].legend(); ax[0,0].grid(alpha=.3)

# (2) Score separation (positive vs negative triple scores)
ax[0,1].plot(ep, history["train_pos"], label="positive")
ax[0,1].plot(ep, history["train_neg"], label="negative")
ax[0,1].fill_between(ep, history["train_neg"], history["train_pos"], alpha=.15)
ax[0,1].set_title("Score separation (larger gap = better)")
ax[0,1].set_xlabel("epoch"); ax[0,1].legend(); ax[0,1].grid(alpha=.3)

# (3) Modality-gate evolution per relation
gate_hist = np.array(history["gates"])          # [epochs, num_relations]
for r in range(gate_hist.shape[1]):
    ax[1,0].plot(ep, gate_hist[:, r], lw=1, label=id2relation[r])
ax[1,0].axhline(0.5, ls="--", c="grey", lw=.8)
ax[1,0].set_ylim(-.02, 1.02)
ax[1,0].set_title("Modality gate evolution (1=text, 0=image)")
ax[1,0].set_xlabel("epoch")
ax[1,0].legend(fontsize=6, ncol=2, loc="center right")

# (4) Generalization gap (overfitting check)
gap = np.array(history["val_loss"]) - np.array(history["train_loss"])
ax[1,1].plot(ep, gap, color="#b5179e")
ax[1,1].axhline(0, ls="--", c="grey", lw=.8)
ax[1,1].set_title("Generalization gap (val - train loss)")
ax[1,1].set_xlabel("epoch"); ax[1,1].grid(alpha=.3)

plt.tight_layout()
plt.savefig("training_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_dynamics.png")


In [ ]:
@torch.no_grad()
def evaluate_link_prediction(model, triples, num_entities, batch_size=256, max_eval=10000):
    """Compute MRR and Hits@K for tail prediction."""
    model.eval()

    if len(triples) > max_eval:
        idx = np.random.choice(len(triples), max_eval, replace=False)
        eval_triples = triples[idx]
    else:
        eval_triples = triples

    ranks = []
    all_entity_ids = torch.arange(num_entities, device=device)

    for i in range(0, len(eval_triples), batch_size):
        batch = eval_triples[i:i+batch_size]
        heads = torch.tensor(batch[:, 0], device=device)
        rels = torch.tensor(batch[:, 1], device=device)
        tails = torch.tensor(batch[:, 2], device=device)

        for j in range(len(batch)):
            h = heads[j].expand(num_entities)
            r = rels[j].expand(num_entities)

            # Score all entities as tail
            scores = model(h, r, all_entity_ids)

            # Rank of true tail
            true_score = scores[tails[j]].item()
            rank = (scores >= true_score).sum().item()
            ranks.append(rank)

    ranks = np.array(ranks, dtype=np.float32)

    mrr = np.mean(1.0 / ranks)
    hits1 = np.mean(ranks <= 1)
    hits3 = np.mean(ranks <= 3)
    hits10 = np.mean(ranks <= 10)

    return {"MRR": mrr, "Hits@1": hits1, "Hits@3": hits3, "Hits@10": hits10}


# Load best model
model.load_state_dict(torch.load("modgated_rotate_best.pt"))

print("Evaluating on validation set...")
metrics = evaluate_link_prediction(model, val_triples, num_entities, max_eval=5000)

print(f"\nLink Prediction Results:")
print(f"  MRR:     {metrics['MRR']:.4f}")
print(f"  Hits@1:  {metrics['Hits@1']:.4f}")
print(f"  Hits@3:  {metrics['Hits@3']:.4f}")
print(f"  Hits@10: {metrics['Hits@10']:.4f}")

# Per-relation evaluation
print("\nPer-relation metrics:")
for rid in range(num_relations):
    rel_mask = val_triples[:, 1] == rid
    if rel_mask.sum() < 10:
        continue
    rel_triples = val_triples[rel_mask]
    rel_metrics = evaluate_link_prediction(model, rel_triples, num_entities, max_eval=1000)
    print(f"  {id2relation[rid]:25s} MRR={rel_metrics['MRR']:.4f}  "
          f"H@1={rel_metrics['Hits@1']:.4f}  H@10={rel_metrics['Hits@10']:.4f}")

In [ ]:
print("\nEvaluating on TEST set...")
test_metrics = evaluate_link_prediction(model, test_triples, num_entities, max_eval=5000)

print(f"\nTest Set Results:")
print(f"  MRR:     {test_metrics['MRR']:.4f}")
print(f"  Hits@1:  {test_metrics['Hits@1']:.4f}")
print(f"  Hits@3:  {test_metrics['Hits@3']:.4f}")
print(f"  Hits@10: {test_metrics['Hits@10']:.4f}")

# Per-relation test evaluation (stored for the bar chart below)
per_rel_mrr = {}
print("\nPer-relation test metrics:")
for rid in range(num_relations):
    rel_mask = test_triples[:, 1] == rid
    if rel_mask.sum() < 10:
        continue
    rel_triples = test_triples[rel_mask]
    rel_metrics = evaluate_link_prediction(model, rel_triples, num_entities, max_eval=1000)
    per_rel_mrr[id2relation[rid]] = rel_metrics["MRR"]
    print(f"  {id2relation[rid]:25s} MRR={rel_metrics['MRR']:.4f}  "
          f"H@1={rel_metrics['Hits@1']:.4f}  H@10={rel_metrics['Hits@10']:.4f}")

print("\nVal vs Test comparison:")
for metric in ["MRR", "Hits@1", "Hits@3", "Hits@10"]:
    diff = abs(metrics[metric] - test_metrics[metric])
    print(f"  {metric}: val={metrics[metric]:.4f} test={test_metrics[metric]:.4f} diff={diff:.4f}")
    if diff > 0.05:
        print(f"    WARNING: Large gap — possible overfitting")


In [ ]:
# ============================================================
# Phase B export: DISENTANGLED channels for IRGNN (Phase C)
#   - base / text / image saved SEPARATELY and MASKED
#   - per-relation gates (alpha_r) for gate-routed message passing
#   - modality masks so IRGNN knows which items lack a channel
# ============================================================
import numpy as np, json

model.load_state_dict(torch.load("modgated_rotate_best.pt"))
model.eval()

BATCH = 8192
all_ids = torch.arange(num_entities, device=device)

struct_ch = np.zeros((num_entities, EMBEDDING_DIM), dtype=np.float32)
text_ch   = np.zeros((num_entities, EMBEDDING_DIM), dtype=np.float32)
image_ch  = np.zeros((num_entities, EMBEDDING_DIM), dtype=np.float32)

with torch.no_grad():
    for s in range(0, num_entities, BATCH):
        e = min(s + BATCH, num_entities)
        eid = all_ids[s:e]
        ht = model.has_text[eid].unsqueeze(-1).float()
        hi = model.has_image[eid].unsqueeze(-1).float()
        struct_ch[s:e] = model.entity_emb(eid).cpu().numpy()
        text_ch[s:e]   = (model.text_proj(model.text_emb[eid]) * ht).cpu().numpy()
        image_ch[s:e]  = (model.image_proj(model.image_emb[eid]) * hi).cpu().numpy()

# per-relation gates alpha_r (this is the routing vector Phase C consumes)
gates = torch.sigmoid(model.gate_weights.weight).squeeze(-1).detach().cpu().numpy()
mean_gate = float(gates.mean())

# fused single-vector embedding (mean-gate blend) for backward compatibility
ht_all = model.has_text.unsqueeze(-1).float().cpu().numpy()
hi_all = model.has_image.unsqueeze(-1).float().cpu().numpy()
has_both, text_only = ht_all * hi_all, ht_all * (1 - hi_all)
fused = struct_ch + has_both * (mean_gate * text_ch + (1 - mean_gate) * image_ch) + text_only * text_ch

np.save("rotateE_struct_embeddings.npy", struct_ch)     # h_struct channel
np.save("rotateE_text_projected.npy",   text_ch)        # h_text channel (MASKED)
np.save("rotateE_image_projected.npy",  image_ch)       # h_img  channel (MASKED)
np.save("rotateE_entity_embeddings.npy", fused)         # fused (backward compat)
np.save("rotateE_relation_embeddings.npy", model.relation_emb.weight.detach().cpu().numpy())
np.save("modality_gates.npy", gates)                    # alpha_r for gate routing
np.save("has_text_mask.npy",  model.has_text.cpu().numpy())
np.save("has_image_mask.npy", model.has_image.cpu().numpy())

json.dump({id2relation[i]: float(gates[i]) for i in range(num_relations)},
          open("modality_gates.json", "w"), indent=2)

torch.save({
    "model_state_dict": model.state_dict(),
    "num_entities": num_entities, "num_relations": num_relations,
    "dim": EMBEDDING_DIM, "gamma": GAMMA,
    "entity2id": entity2id, "relation2id": relation2id,
}, "modgated_rotate_checkpoint.pt")

print("Saved disentangled channels + gates + masks for Phase C.")
print(f"Mean gate: {mean_gate:.4f}")


In [ ]:
# ============================================================
# SANITY CHECKS for the disentangled export
# ============================================================
import numpy as np

struct = np.load("rotateE_struct_embeddings.npy")
text   = np.load("rotateE_text_projected.npy")
image  = np.load("rotateE_image_projected.npy")
fused  = np.load("rotateE_entity_embeddings.npy")
gates  = np.load("modality_gates.npy")
ht     = np.load("has_text_mask.npy").astype(np.float32)[:, None]
hi     = np.load("has_image_mask.npy").astype(np.float32)[:, None]

print("Entities:", struct.shape[0], " (expect ~617K, NOT ~1.04M)")
assert struct.shape[0] < 750_000, "GHOST NODES — retrain on the FIXED KG first!"

for name, arr in [("struct", struct), ("text", text), ("image", image), ("fused", fused)]:
    n = np.linalg.norm(arr, axis=1)
    print(f"  {name:6s} shape={arr.shape} nan={np.isnan(arr).any()} "
          f"norm_mean={n.mean():.4f} zero_rows={int((n < 1e-6).sum())}")
print("  -> text/image zero_rows should be ~#aux entities (~28K), NOT ~600K.")

# (1) channels must reconstruct the model embedding exactly (eval mode)
model.eval()
rng = np.random.default_rng(0)
se = rng.integers(0, num_entities, size=4000)
sr = rng.integers(0, num_relations, size=4000)
with torch.no_grad():
    ref = model.get_entity_embedding(
        torch.tensor(se, device=device),
        torch.tensor(sr, device=device)).cpu().numpy()
g  = gates[sr][:, None]
hb = ht[se] * hi[se]
to = ht[se] * (1 - hi[se])
recon = struct[se] + hb * (g * text[se] + (1 - g) * image[se]) + to * text[se]
max_err = float(np.abs(recon - ref).max())
print(f"\nReconstruction max abs error: {max_err:.2e}  (must be < 1e-4)")
assert max_err < 1e-4, "Channels do NOT reconstruct the model embedding!"

# (2) gate diversity + the emergent-gates table
print(f"\nGate std: {gates.std():.4f}  ({'diverse — good' if gates.std() > 0.05 else 'COLLAPSED — check'})")
for i in np.argsort(-gates):
    print(f"  {id2relation[i]:25s} {gates[i]:.4f}")
print("\nExpected: has_brand & parent_category TEXT-dominant (>0.6); has_color image-dominant (<0.4).")
print("If has_brand ~0.03 and belongs_to ~0.02 -> you trained on the GHOST-NODE graph.")


# Headline figures: modality gates + per-relation MRR

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

gates = np.load("modality_gates.npy")
order = np.argsort(gates)
names = [id2relation[i] for i in order]

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# (left) learned modality gates
colors = ["#2a9d8f" if g >= 0.5 else "#e76f51" for g in gates[order]]
ax[0].barh(names, gates[order], color=colors)
ax[0].axvline(0.5, ls="--", c="grey")
ax[0].set_xlim(0, 1)
ax[0].set_xlabel("gate alpha_r  (0 = image, 1 = text)")
ax[0].set_title("Learned modality gates per relation")
for i, gv in enumerate(gates[order]):
    ax[0].text(gv + .01, i, f"{gv:.2f}", va="center", fontsize=8)

# (right) per-relation link-prediction MRR (from the test-eval cell)
try:
    rels = sorted(per_rel_mrr, key=per_rel_mrr.get)
    ax[1].barh(rels, [per_rel_mrr[r] for r in rels], color="#457b9d")
    ax[1].set_xlabel("MRR"); ax[1].set_title("Per-relation link prediction (test)")
    for i, r in enumerate(rels):
        ax[1].text(per_rel_mrr[r] + .005, i, f"{per_rel_mrr[r]:.2f}", va="center", fontsize=8)
except NameError:
    ax[1].text(.5, .5, "run the test-eval cell first\n(per_rel_mrr not found)",
               ha="center", va="center"); ax[1].axis("off")

plt.tight_layout()
plt.savefig("gates_and_mrr.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved gates_and_mrr.png")
